In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.options.display.max_columns = 120

PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
DATA_DIR = PROJECT_ROOT / "data"
LOAD_FILE = DATA_DIR / "interim" / "eia930" / "pjm_hourly_load_2023.parquet"
WEATHER_FILE = (
    DATA_DIR
    / "interim"
    / "era5"
    / "era5_pjm_box_hourly_features_2023_0101_0107.parquet"
)
FEATURE_DIR = DATA_DIR / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
JOINED_FILE = FEATURE_DIR / "pjm_load_era5_hourly_2023_0101_0107.parquet"

LOAD_FILE, WEATHER_FILE, JOINED_FILE

# Join PJM Load with ERA5 Weather

Purpose: create a first joined hourly dataset for weather-driven demand exploration and baseline forecasting.

This notebook assumes the prior notebooks have created:
- `data/interim/eia930/pjm_hourly_load_2023.parquet`
- `data/interim/era5/era5_pjm_box_hourly_features_2023_0101_0107.parquet`

Research guardrails:
- The join key is UTC timestamp.
- All future model splits must be walk-forward.
- Realized ERA5 weather at the target hour is non-causal for many real forecasting horizons. Use this joined dataset first for relationship analysis, then construct horizon-specific non-leaky features.

In [ ]:
if not LOAD_FILE.exists():
    raise FileNotFoundError(f"Run eia930_pjm_load_eda.ipynb first: {LOAD_FILE}")
if not WEATHER_FILE.exists():
    raise FileNotFoundError(
        f"Run era5_hourly_weather_exploration.ipynb first: {WEATHER_FILE}"
    )

load = pd.read_parquet(LOAD_FILE)
weather = pd.read_parquet(WEATHER_FILE)
load["timestamp_utc"] = pd.to_datetime(load["timestamp_utc"], utc=True)
weather["timestamp_utc"] = pd.to_datetime(weather["timestamp_utc"], utc=True)

load.shape, weather.shape

## Join and Validate

In [ ]:
joined = load.merge(weather, on="timestamp_utc", how="inner", validate="one_to_one")
joined = joined.sort_values("timestamp_utc").reset_index(drop=True)

coverage = pd.Series(
    {
        "load_rows": len(load),
        "weather_rows": len(weather),
        "joined_rows": len(joined),
        "load_start": load["timestamp_utc"].min(),
        "load_end": load["timestamp_utc"].max(),
        "weather_start": weather["timestamp_utc"].min(),
        "weather_end": weather["timestamp_utc"].max(),
        "joined_start": joined["timestamp_utc"].min(),
        "joined_end": joined["timestamp_utc"].max(),
    }
)
coverage

In [ ]:
expected = pd.date_range(
    joined["timestamp_utc"].min(), joined["timestamp_utc"].max(), freq="h", tz="UTC"
)
missing_hours = expected.difference(pd.DatetimeIndex(joined["timestamp_utc"]))

integrity = pd.Series(
    {
        "missing_joined_hours": len(missing_hours),
        "duplicate_joined_hours": joined["timestamp_utc"].duplicated().sum(),
        "any_null_load": joined["load_mw"].isna().any(),
        "weather_null_cells": joined.drop(
            columns=["timestamp_utc", "period_raw", "load_mw"], errors="ignore"
        )
        .isna()
        .sum()
        .sum(),
    }
)
integrity

## Calendar and Weather-Demand Features

These features are for exploration. Horizon-specific forecasting features should be generated separately so leakage assumptions are explicit.

In [ ]:
features = joined.copy()
ts = features["timestamp_utc"]
features["hour_utc"] = ts.dt.hour
features["dayofweek"] = ts.dt.dayofweek
features["month"] = ts.dt.month
features["is_weekend"] = features["dayofweek"].isin([5, 6])

if "temperature_2m_f" in features:
    base = 65.0
    features["cooling_degree_f"] = np.maximum(features["temperature_2m_f"] - base, 0.0)
    features["heating_degree_f"] = np.maximum(base - features["temperature_2m_f"], 0.0)
    features["extreme_heat"] = features["temperature_2m_f"] >= features[
        "temperature_2m_f"
    ].quantile(0.95)
    features["extreme_cold"] = features["temperature_2m_f"] <= features[
        "temperature_2m_f"
    ].quantile(0.05)

features.to_parquet(JOINED_FILE, index=False)
features.head(), features.shape

## Relationship Diagnostics

In [ ]:
if "temperature_2m_f" in features:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(features["temperature_2m_f"], features["load_mw"], s=8, alpha=0.25)
    ax.set_title("PJM load versus ERA5 area-mean temperature")
    ax.set_xlabel("Temperature (F)")
    ax.set_ylabel("Load (MW)")
    plt.show()

In [ ]:
if "temperature_2m_f" in features:
    temp_bins = pd.cut(features["temperature_2m_f"], bins=30)
    temp_response = features.groupby(temp_bins, observed=True)["load_mw"].agg(
        ["mean", "median", "count"]
    )
    temp_response.head(), temp_response.tail()

In [ ]:
if "temperature_2m_f" in features:
    temp_response = temp_response.reset_index()
    temp_response["temp_mid_f"] = temp_response["temperature_2m_f"].apply(
        lambda interval: interval.mid
    )
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(
        temp_response["temp_mid_f"], temp_response["mean"], marker="o", linewidth=1.5
    )
    ax.set_title("Average load response by temperature bin")
    ax.set_xlabel("Temperature bin midpoint (F)")
    ax.set_ylabel("Mean load (MW)")
    plt.show()

## Seasonal and Extreme-Regime Slices

In [ ]:
season_map = {
    12: "winter",
    1: "winter",
    2: "winter",
    3: "spring",
    4: "spring",
    5: "spring",
    6: "summer",
    7: "summer",
    8: "summer",
    9: "fall",
    10: "fall",
    11: "fall",
}
features["season"] = features["month"].map(season_map)

slice_cols = ["load_mw"]
if "temperature_2m_f" in features:
    slice_cols.append("temperature_2m_f")

features.groupby("season")[slice_cols].describe()

In [ ]:
if {"extreme_heat", "extreme_cold"}.issubset(features.columns):
    regime_summary = pd.concat(
        {
            "extreme_heat": features.loc[
                features["extreme_heat"], "load_mw"
            ].describe(),
            "extreme_cold": features.loc[
                features["extreme_cold"], "load_mw"
            ].describe(),
            "normal_temp": features.loc[
                ~features["extreme_heat"] & ~features["extreme_cold"], "load_mw"
            ].describe(),
        },
        axis=1,
    )
    regime_summary

## Simple Walk-Forward Split Skeleton

This is a split design placeholder, not a final evaluation. It avoids random splits and gives later models a consistent temporal frame.

In [ ]:
split_1 = pd.Timestamp("2023-09-01", tz="UTC")
split_2 = pd.Timestamp("2023-11-01", tz="UTC")

train = features[features["timestamp_utc"] < split_1]
validation = features[
    (features["timestamp_utc"] >= split_1) & (features["timestamp_utc"] < split_2)
]
test = features[features["timestamp_utc"] >= split_2]

pd.DataFrame(
    {
        "rows": [len(train), len(validation), len(test)],
        "start": [
            train["timestamp_utc"].min(),
            validation["timestamp_utc"].min(),
            test["timestamp_utc"].min(),
        ],
        "end": [
            train["timestamp_utc"].max(),
            validation["timestamp_utc"].max(),
            test["timestamp_utc"].max(),
        ],
    },
    index=["train", "validation", "test"],
)

## Next Research Questions

- Should the first target be hourly load, next-day peak, or day-ahead hourly load profile?
- What weather feature availability is realistic for the selected target horizon?
- Which regime labels should be added first: holidays, temperature tails, named storms, or outage/demand-response events?
- Do baseline errors concentrate in specific seasons or extreme temperature regimes?